In [48]:
import sys 
sys.path.append("..")
from pathlib import Path

DOCS_DIR = Path("../data/documents")

In [49]:
# Cell 2: load documents

docs=[]
for i in sorted(DOCS_DIR.glob("*.txt")):
    docs.append({"source":i.name,"text":i.read_text(encoding="utf-8")})

print(len(docs),"documents loaded")    

6 documents loaded


In [50]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
chunks=[]
for d in docs :
    for i,piece in enumerate(splitter.split_text(d["text"])):
       chunks.append({"id":f"{d['source']}-{i}",
       "text":piece,
       "source":d["source"]})

print(len(chunks),"chunks ")
for c in chunks[:3]:
    print("id",c["id"],c["text"])


20 chunks 
id account_security.txt-0 GubbisCart Account Security Policy
id account_security.txt-1 Regularly update your password every 90 days to maintain Security
Password should be of minimum 6 characters 
Password should contain one Capital letter, one small letter,one Number and special characters
Password should not be shared with anyone
id account_security.txt-2 Password should not be shared with anyone
if you suspecious about any account activity please change your password immediately and contact customer support
For password reset raise request with customer support
Account will be locked after 3 failed attempts
2FA Autentication is supported


In [51]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
chunk_vectors=embeddings.embed_documents(c["text"] for c in chunks)

print(len(chunk_vectors),len(chunk_vectors[0]))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

20 384


In [52]:
import chromadb

client =chromadb.Client()

collections=client.create_collection(
                   name="gubbiscart1",
                   metadata={"hnsw:space":"cosine"}

)

collections.add(ids=[c["id"] for c in chunks],
                documents=[c["text"] for c in chunks],
                embeddings=chunk_vectors,
                metadatas=[{"source":c["source"]} for c in chunks]
)

print(collections.count)


<bound method Collection.count of Collection(name=gubbiscart1)>


In [61]:
from langchain_huggingface import HuggingFaceEmbeddings


def retrieve(question,top_k=3):
    q_vec=embeddings.embed_query(question)
    res=collections.query(query_embeddings=[q_vec],n_results=top_k)
    return [
         {"text":t,"source":m["source"],"similarity":round(1-d,3)}
         for  t,m,d in zip(res["documents"][0],res["metadatas"][0],res["distances"][0])
    ]


for i in retrieve("what is miniumum length of password"):
  print(i["text"][:20].replace("\n"," "),i["source"],i["similarity"])

Regularly update you account_security.txt 0.46
Password should not  account_security.txt 0.315
We collect personal  privacy_policy.txt 0.134


In [68]:
def build_prompt(question,retrieved):
    context="\n\n".join(f"[source:{r['source']}]\n{r['text']}"  for r in retrieved)
    return f"""You are a helpful assistant for Gubbiscart
    Answer the questions  using the given context 
    If the context does not contain the answers then reply exactly: "I do not have enough information "
    Be concise 
    
    Context:{context}
    Question:{question}
    Answer:"""

q="what is the refund policy?"
print(build_prompt(q,retrieve(q)))


You are a helpful assistant for Gubbiscart
    Answer the questions  using the given context 
    If the context does not contain the answers then reply exactly: "I do not have enough information "
    Be concise 

    Context:[source:refundpolicy.txt]
GubbisCart Refund Policy

[source:refundpolicy.txt]
Customer can request refund within 30 Days of Delivery
Refund is applicable only for damaged products
Screenshot of damaged product should be sent before raising refund request
Refunds are Processed within 7 business Days to original Payment mode
Digital Downloads are not eligible for Refund

[source:refundpolicy.txt]
Digital Downloads are not eligible for Refund
Damaged items are eligible for a full refund including shipping costs, if reported within 48 hours of delivery.
    Question:what is the refund policy?
    Answer:


In [69]:
from src.llm import get_llm

llm=get_llm()


def ask(question,top_k=3):
    retrieved=retrieve(question,top_k)
    prompt=build_prompt(question,retrieved)
    answer=llm.invoke(prompt).content.strip()
    return {"question":question,"answer":answer,
             "sources":[r['source'] for r in retrieved],
             "contexts":[r['text'] for r in retrieved]
    }

question="what is my name?"
ans=ask(question)
print(f"answer:{ans['answer']}")
print(ans['sources'])




answer:I do not have enough information
['privacy_policy.txt', 'privacy_policy.txt', 'warranty_policy.txt']
